In [1]:
import  gym
import numpy as np
import random
import math
import matplotlib.pyplot as plt

In [2]:
iterations = 1
episodes = 50000

gamma = 0.95 
alpha = 0.2
epsilon = 1.0
min_epsilon = 0.1
buckets = (5, 5, 5, 5, 5, 5, 2, 2)
average_window = 500
average_rewards_list = []

In [3]:
env = gym.make("LunarLander-v2")
observation = env.observation_space
n = env.action_space.n

In [4]:
ranges = list(zip(env.observation_space.low, env.observation_space.high))
print(ranges)
ranges[0] = [-1, 1]
ranges[1] = [-1, 1]
ranges[2] = [-1, 1]
ranges[3] = [-1, 1]
ranges[4] = [-1, 1]
ranges[5] = [-1, 1]
ranges[6] = [0, 1]
ranges[7] = [0, 1]

[(-1.5, 1.5), (-1.5, 1.5), (-5.0, 5.0), (-5.0, 5.0), (-3.1415927, 3.1415927), (-5.0, 5.0), (-0.0, 1.0), (-0.0, 1.0)]


In [5]:
Q_table = np.zeros(buckets + (n,))

print(Q_table.shape)
print(Q_table)

(5, 5, 5, 5, 5, 5, 2, 2, 4)
[[[[[[[[[0. 0. 0. 0.]
        [0. 0. 0. 0.]]

       [[0. 0. 0. 0.]
        [0. 0. 0. 0.]]]


      [[[0. 0. 0. 0.]
        [0. 0. 0. 0.]]

       [[0. 0. 0. 0.]
        [0. 0. 0. 0.]]]


      [[[0. 0. 0. 0.]
        [0. 0. 0. 0.]]

       [[0. 0. 0. 0.]
        [0. 0. 0. 0.]]]


      [[[0. 0. 0. 0.]
        [0. 0. 0. 0.]]

       [[0. 0. 0. 0.]
        [0. 0. 0. 0.]]]


      [[[0. 0. 0. 0.]
        [0. 0. 0. 0.]]

       [[0. 0. 0. 0.]
        [0. 0. 0. 0.]]]]



     [[[[0. 0. 0. 0.]
        [0. 0. 0. 0.]]

       [[0. 0. 0. 0.]
        [0. 0. 0. 0.]]]


      [[[0. 0. 0. 0.]
        [0. 0. 0. 0.]]

       [[0. 0. 0. 0.]
        [0. 0. 0. 0.]]]


      [[[0. 0. 0. 0.]
        [0. 0. 0. 0.]]

       [[0. 0. 0. 0.]
        [0. 0. 0. 0.]]]


      [[[0. 0. 0. 0.]
        [0. 0. 0. 0.]]

       [[0. 0. 0. 0.]
        [0. 0. 0. 0.]]]


      [[[0. 0. 0. 0.]
        [0. 0. 0. 0.]]

       [[0. 0. 0. 0.]
        [0. 0. 0. 0.]]]]



     [[[[0. 0. 0. 0.]
      

In [6]:
def discretize_state(state):
    bucket_indexes = []
    for i in range(len(state)):
        if state[i] <= ranges[i][0]:
            bucket_index = 0
        elif state[i] >= ranges[i][1]:
            bucket_index = buckets[i] - 1
        else:
            bound_width = ranges[i][1] - ranges[i][0]
            offset = (buckets[i] - 1) * ranges[i][0] / bound_width
            scaling = (buckets[i] - 1) / bound_width
            bucket_index = int(round(scaling * state[i] - offset))
        bucket_indexes.append(bucket_index)
    return tuple(bucket_indexes)

In [7]:
def change_epsilon(episode):
    return max(min_epsilon, min(epsilon, 1.0 - math.log10((episode + 1) / (episodes * 0.1))))

def e_greedy_algorithm(Q_table, discrete_state, epsilon):
    if np.random.random() <= epsilon:
        return env.action_space.sample()  
    else:
        return np.argmax(Q_table[discrete_state])  

In [8]:
def q_learning(Q_table):
    global epsilon 
    global alpha
    rewards = []
    max_env_steps = env._max_episode_steps
    num = 0
    for episode in range(1, episodes):
        total_reward = 0
        state = env.reset()
        discrete_state = discretize_state(state[0])
        for step in range(max_env_steps):
            action = e_greedy_algorithm(Q_table, discrete_state, epsilon)
            new_state, reward, done, info, _ = env.step(action)
            discrete_new_state = discretize_state(new_state)

            best_next_action = np.argmax(Q_table[discrete_new_state])
            td_target = reward + gamma * Q_table[discrete_new_state][best_next_action]
            td_error = td_target - Q_table[discrete_state][action]
            Q_table[discrete_state][action] += alpha * td_error
            
            total_reward += reward
            discrete_state = discrete_new_state

            if done:
                break
            
        if total_reward >= 200:
            num += 1
        
        if (episode) % 100 == 0 and epsilon > min_epsilon:
            epsilon -= 0.0025
        
        if (episode) % 100 == 0 and alpha > 0.01:
            alpha -= 0.00125

        rewards.append(total_reward)
        if episode % 100 == 0:
            print("Episode {}, epsilon {:5.4f}, reward {:6.2f}, num {}".format(episode, epsilon, total_reward, num))
            np.save('q_table.npy', Q_table)

    print("FINAL Episode {}, epsilon {:5.4f}, reward {:6.2f}, num {}".format(episode, epsilon, total_reward, num))
    return Q_table, rewards

In [9]:
import pandas as pd
import matplotlib.ticker as mticker


train=True
if train:
  Q_table, rewards = q_learning(Q_table)
  average_rewards = [np.mean(rewards[i:i + average_window]) for i in range(0, episodes, average_window)]
  average_rewards_list.append(average_rewards)
  lower_limit = 0
  upper_limit = len(rewards)
  window_size = (upper_limit) // 10
  points = rewards[lower_limit:upper_limit]
  episode_num = [x for x in range(lower_limit, upper_limit)]
  rolling_mean = pd.DataFrame(points).rolling(window_size).mean()

  plt.figure(figsize=(10, 7), facecolor="white")
  plt.plot(episode_num, points, linewidth=1, color="cyan")
  plt.plot(episode_num, rolling_mean, linewidth=2, color="magenta")

# Plotting the average rewards
  text_color = "black"
  ax = plt.gca()
  ax.set_facecolor("black")
  plt.grid()
  plt.xlabel("Episode", color=text_color, fontsize=30)
  plt.ylabel("Total Points", color=text_color, fontsize=30)
  yNumFmt = mticker.StrMethodFormatter("{x:,}")
  ax.yaxis.set_major_formatter(yNumFmt)
  ax.tick_params(axis="x", colors=text_color)
  ax.tick_params(axis="y", colors=text_color)
  plt.show()
  np.save('q_table.npy', Q_table)
else:
  Q_table = np.load('q_table.npy')



/Users/davemag9/anaconda3/envs/vscodeenv/lib/python3.11/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


Episode 100, epsilon 0.9975, reward -96.71, num 0
Episode 200, epsilon 0.9950, reward -128.65, num 0
Episode 300, epsilon 0.9925, reward -100.96, num 0
Episode 400, epsilon 0.9900, reward -296.98, num 0
Episode 500, epsilon 0.9875, reward -158.75, num 0
Episode 600, epsilon 0.9850, reward -350.99, num 0
Episode 700, epsilon 0.9825, reward -351.84, num 0
Episode 800, epsilon 0.9800, reward -17.66, num 0
Episode 900, epsilon 0.9775, reward -80.35, num 0
Episode 1000, epsilon 0.9750, reward -115.07, num 0
Episode 1100, epsilon 0.9725, reward -141.84, num 0
Episode 1200, epsilon 0.9700, reward -86.17, num 0
Episode 1300, epsilon 0.9675, reward -141.65, num 0
Episode 1400, epsilon 0.9650, reward -111.89, num 0
Episode 1500, epsilon 0.9625, reward -150.35, num 0
Episode 1600, epsilon 0.9600, reward -82.96, num 0
Episode 1700, epsilon 0.9575, reward -81.23, num 0
Episode 1800, epsilon 0.9550, reward -200.94, num 0
Episode 1900, epsilon 0.9525, reward -121.60, num 0
Episode 2000, epsilon 0.950

KeyboardInterrupt: 

In [ ]:
env_test = gym.make('LunarLander-v2', render_mode="human")

successes = 0
total_episodes = 100

for i in range(total_episodes):
    total_reward = 0
    state = env_test.reset()[0]
    discrete_state_test = discretize_state(state)
    done = False
    print("Episode: ", i)
    while not done:
       
        action = np.argmax(Q_table[discrete_state_test])
        next_state, reward, done, info, _ = env_test.step(action)
        discrete_state_test = discretize_state(next_state)
        total_reward+=reward
        if done:
            print("Reward: ", total_reward)
            if total_reward >= 200: 
                successes += 1
env_test.close()

Episode:  0


c:\Users\ACER\lander_new\.conda\lib\site-packages\gym\utils\passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


Reward:  218.959329130088
Episode:  1
Reward:  259.5885652592477
Episode:  2
Reward:  239.28187629180638
Episode:  3
Reward:  191.92273388933336
Episode:  4
Reward:  252.9284802681973
Episode:  5
Reward:  245.79260671329996
Episode:  6
Reward:  246.48968528279977
Episode:  7
Reward:  221.81566902470576
Episode:  8
Reward:  241.05975264861968
Episode:  9
Reward:  189.34114151657639
Episode:  10
Reward:  -28.73276177570297
Episode:  11
Reward:  266.38171972358026
Episode:  12
Reward:  244.5766066753169
Episode:  13
Reward:  243.1908394089934
Episode:  14
Reward:  232.9936751028511
Episode:  15
Reward:  252.97857829484195
Episode:  16
Reward:  145.2490293838918
Episode:  17
Reward:  -143.93926228341388
Episode:  18
Reward:  245.53787778061113
Episode:  19
Reward:  203.23934977678408
Episode:  20
Reward:  215.04150931617204
Episode:  21
Reward:  244.56294014109582
Episode:  22
Reward:  -36.4023925886338
Episode:  23
Reward:  245.90331987616645
Episode:  24
Reward:  257.5345285342182
Episod

In [ ]:
accuracy = successes / total_episodes * 100
print("Accuracy: {:.2f}%".format(accuracy))
print(successes)

Accuracy: 60.00%
60


In [ ]:
def evaluate_agent(env, max_steps, n_eval_episodes, Q, seed):

  episode_rewards = []
  for episode in range(n_eval_episodes):
    if seed:
      state, _ = env.reset(seed=seed[episode])
      discrete_state = discretize_state(state)
    else:
      state, _ = env.reset()
      discrete_state = discretize_state(state)
    step = 0
    done = False
    total_rewards_ep = 0
   
    for step in range(max_steps):
      # Take the action (index) that have the maximum reward
      action = np.argmax(Q[discrete_state][:])
      new_state, reward, done,_, info = env.step(action)
      discrete_new_state = discretize_state(new_state)
      total_rewards_ep += reward
       
      if done:
        break
      discrete_state = discrete_new_state
    episode_rewards.append(total_rewards_ep)
  mean_reward = np.mean(episode_rewards)

  return mean_reward

In [ ]:
# Evaluate our Agent
eval_seed=[]
mean_reward, std_reward = evaluate_agent(env, 1000, 100, Q_table, eval_seed)
print(f"Mean_reward={mean_reward:.2f}")

Mean_reward=162.61


In [ ]:
every_100th_episode = episode_num[::100]
every_100th_points = points[::100]

plt.figure(figsize=(10, 7), facecolor="white")
plt.plot(every_100th_episode, every_100th_points, linewidth=1, color="cyan")
plt.plot(episode_num, rolling_mean, linewidth=2, color="magenta")
plt.xlabel("Episode Number")
plt.ylabel("Points")
plt.title("Every 100th Episode Points with Rolling Mean")
plt.grid(True)
plt.show()

NameError: name 'episode_num' is not defined